In [2]:
%load_ext autoreload
%autoreload 2
import pprint, json, math, os, sys, camelot
# sys.path.append(os.path.abspath(dir_path))
import subprocess

import fitz, pdfplumber, ocrmypdf, pprint
import pandas as pd
import numpy as np
from collections import defaultdict
from app.parse_table import TableParser
from app.utils import Helper, PDFTableExtractor

helper = Helper()

In [1]:
import fitz
import pandas as pd


# =========================
# 1. ROW EXTRACTION (Y-axis)
# =========================
def extract_rows(page, bbox=None):
    items = []

    for block in page.get_text("dict")["blocks"]:
        if "lines" not in block:
            continue

        for line in block["lines"]:
            for span in line["spans"]:
                x0, y0, x1, y1 = span["bbox"]
                text = span["text"].strip()

                if not text:
                    continue

                if bbox:
                    bx0, by0, bx1, by1 = bbox
                    if not (x0 >= bx0 and y0 >= by0 and x1 <= bx1 and y1 <= by1):
                        continue

                items.append({
                    "x0": x0, "y0": y0,
                    "x1": x1, "y1": y1,
                    "text": text,
                    "x_center": (x0 + x1) / 2,
                    "y_center": (y0 + y1) / 2,
                    "height": y1 - y0
                })

    if not items:
        return []

    items.sort(key=lambda x: x["y_center"])

    heights = [i["height"] for i in items]
    avg_height = sum(heights) / len(heights)
    y_threshold = avg_height * 0.5

    rows = []
    current_row = [items[0]]

    for i in range(1, len(items)):
        if abs(items[i]["y_center"] - current_row[-1]["y_center"]) < y_threshold:
            current_row.append(items[i])
        else:
            rows.append(current_row)
            current_row = [items[i]]

    rows.append(current_row)

    return rows


# =========================
# 2. ANCHOR CUT (ROW LEVEL)
# =========================
def apply_anchor_cut_rows(rows, top_left=None, top_right=None):
    new_rows = []

    for row in rows:
        filtered = []

        for item in row:
            keep = True

            if top_left:
                ax, ay = top_left
                if item["y1"] < ay or item["x1"] < ax:
                    keep = False

            if top_right:
                rx = top_right[0]
                if item["x0"] > rx:
                    keep = False

            if keep:
                filtered.append(item)

        if filtered:
            new_rows.append(filtered)

    return new_rows


# =========================
# 3. COLUMN ASSIGN (X-axis)
# =========================
def assign_columns(rows, x_lines, tol=10):
    table = []

    for row in rows:
        cols = [[] for _ in range(len(x_lines))]

        for item in row:
            assigned = False

            # pass-through (strong signal)
            for idx, lx in enumerate(x_lines):
                if item["x0"] - tol <= lx <= item["x1"] + tol:
                    cols[idx].append(item)
                    assigned = True
                    break

            # proximity fallback
            if not assigned:
                dists = [abs(item["x_center"] - lx) for lx in x_lines]
                idx = dists.index(min(dists))
                cols[idx].append(item)

        # join text
        row_text = []
        for col in cols:
            col_sorted = sorted(col, key=lambda x: x["x0"])
            row_text.append(" ".join(i["text"] for i in col_sorted))

        table.append(row_text)

    return table


# =========================
# 4. MAIN PIPELINE
# =========================
def extract_table(
    pdf_path,
    page_no=0,
    bbox=None,
    x_lines=None,
    top_left=None,
    top_right=None
):
    if not x_lines:
        raise ValueError("x_lines required")

    doc = fitz.open(pdf_path)
    page = doc[page_no]

    # 1. rows
    rows = extract_rows(page, bbox)

    # 2. anchor cut
    rows = apply_anchor_cut_rows(rows, top_left, top_right)

    if not rows:
        return pd.DataFrame()

    # 3. columns
    table = assign_columns(rows, x_lines)

    df = pd.DataFrame(table)

    doc.close()
    return df


# =========================
# 5. SAVE
# =========================
def save_to_excel(df, path):
    df.to_excel(path, index=False)

In [7]:
from app.utils import Helper
import subprocess

helper = Helper()

pdf_path = "SAM.pdf"

masked_pdf = helper.mask_outside_bboxes(
    pdf_path,
    [(183.46, 37.68, 430.81, 787.9)]
)

subprocess.Popen([masked_pdf], shell=True)

df = extract_table(
    pdf_path=masked_pdf,   # 🔥 use masked pdf
    page_no=1,
    bbox=None,             # already masked
    x_lines=[220, 415],    # from your GUI tool
    # top_left=(430, 120),   # optional
    # top_right=(610, 120)   # optional
)

# print(df)

save_to_excel(df, "soutput.xlsx")